# 02 · 回测引擎与交易成本

> **学习目标**
> 1. 彻底理解「收盘决策 → 次日开盘成交」这个时序，以及它为什么决定了回测可信度
> 2. 亲手量化**交易成本**对策略的侵蚀——这是 90% 回测失败的真实原因
> 3. 看清两个最常见的隐形杀手：**无谓换手** 与 **停牌不可交易**

> 这一节不讲策略有多赚钱，只讲**回测结果为什么可能是假的**。

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import display

from qlearn.backtest import BacktestEngine, CostModel
from qlearn.data import load_panel, make_synthetic_panel
from qlearn.strategies import create_strategy

warnings.filterwarnings('ignore')

pd.set_option('display.width', 180)
print('环境就绪')

## 1. 引擎的时序：一切可信度的基础

```
第 t 日收盘后 ──> 策略用「截止 t 日（含）」的数据算出目标权重 w_t
第 t+1 日开盘 ──> 按 w_t 以开盘价成交，扣除交易成本
第 t+1 日收盘 ──> 按收盘价重新估值，得到净值 V_{t+1}
```

**为什么必须这样设计**：

1. 你在收盘后才拿得到当天的收盘价，所以「用今天的均线决定今天开盘买什么」是作弊（前视偏差）
2. 引擎内部已经处理了这个延迟，所以**策略代码里不要自己再 `shift(1)`**
   - 再 shift 一次 = 把真实的 1 天延迟变成 2 天，结果会系统性失真

**权重的三种取值语义**

| 取值 | 含义 |
|---|---|
| 正数 | 调仓到该权重（占当时总权益的比例） |
| `0` | 清仓该标的 |
| `NaN` | **维持当前持仓不动**（低频率调仓策略专用） |

In [ ]:
USE_REAL_DATA = True
SYMBOLS = ['600519', '000858', '601318', '600036', '000001']
START, END = '2020-01-01', '2024-12-31'

try:
    panel = load_panel(SYMBOLS, start=START, end=END) if USE_REAL_DATA else None
    if panel is None:
        raise RuntimeError('已关闭真实数据')
    data_source = '真实 A 股'
except Exception as exc:
    print(f'[降级] {type(exc).__name__}: {str(exc)[:100]}')
    panel = make_synthetic_panel(SYMBOLS, start=START, end=END, seed=42)
    data_source = '合成数据（无现实意义）'

print(f'数据源: {data_source}｜{panel.n_symbols} 个标的 x {panel.n_dates} 个交易日')

In [ ]:
strategy = create_strategy('ma_cross', fast=10, slow=30)
weights = strategy.generate_weights(panel)

engine = BacktestEngine(initial_capital=1_000_000.0)
result = engine.run(weights, panel, label=strategy.label)

print(result.describe())

## 2. 交易成本：A 股到底收哪些钱？

| 项目 | 费率 | 方向 |
|---|---|---|
| 佣金 | 万分之 2.5（主流 1~3） | 双边，**单笔最低 5 元** |
| 印花税 | 万分之 5 | **仅卖出**（2023-08-28 起由千分之 1 减半） |
| 过户费 | 十万分之 1 | 双边 |
| 滑点 | 假设 2 个基点 | 双边，无法精确建模 |

**注意「单笔最低 5 元」**：这是小资金的高频策略的致命伤。若每笔只成交 5000 元，
万分之 2.5 的佣金只有 1.25 元，但会被抬到 5 元 —— 实际费率达到**万分之一**，是名义费率的 4 倍。

In [ ]:
print('当前成本模型:')
display(engine.cost_model.describe())

print()
print('最低佣金对不同成交金额的实际影响:')
for notional in [2_000, 5_000, 20_000, 100_000, 1_000_000]:
    cost = engine.cost_model.cost(notional, 'buy')
    print(f'  买入 {notional:>9,} 元 -> 成本 {cost:>8.2f} 元，实际费率 {cost / notional:.4%}')

### 2.1 毛收益 vs 净收益：成本到底吃掉多少？

用**完全相同的权重**跑两次回测，只改成本模型，就能干净地量化成本拖累。

In [ ]:
engine_gross = BacktestEngine(initial_capital=1_000_000.0, cost_model=CostModel.zero())
result_gross = engine_gross.run(weights, panel, label='双均线（零成本）')

print('成本影响对照:')
print(result.cost_impact().to_frame().round(4).to_string())

print()
gap = result_gross.metrics()['累计收益'] - result.metrics()['累计收益']
print(f'策略换手带来 {gap:.2%} 的收益侵蚀，全部来自交易成本。')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

for res, color, ls in (
    (result_gross, '#8c8c8c', '--'),
    (result, '#1f4e79', '-'),
):
    curve = res.equity / res.equity.iloc[0]
    ax1.plot(curve.index, curve, color=color, linestyle=ls, linewidth=1.5, label=res.label)

ax1.axhline(1.0, color='black', linewidth=0.8, linestyle=':')
ax1.set_title(f'同一策略的毛收益 vs 净收益｜数据源：{data_source}')
ax1.set_ylabel('净值')
ax1.legend(frameon=False)
ax1.grid(alpha=0.3, linestyle='--')
ax1.spines[['top', 'right']].set_visible(False)

cum_cost = result.costs.cumsum()
ax2.fill_between(cum_cost.index, cum_cost.to_numpy(), 0, color='#c0392b', alpha=0.35, linewidth=0)
ax2.plot(cum_cost.index, cum_cost, color='#c0392b', linewidth=1.2)
ax2.set_title('累计交易成本（元）')
ax2.set_ylabel('元')
ax2.grid(alpha=0.3, linestyle='--')
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 3. 隐形杀手一：无谓换手

如果策略**每天都输出同一个目标权重**（比如「持有，权重 0.2」），引擎就会
每天把组合拉回 0.2 —— 这等于**每日再平衡**：涨了减仓、跌了加仓，
换手率能飙到 1000%+/年，而这些交易不带来任何信号价值。

真实趋势策略的做法是「信号不变就一直拿着」。本项目默认启用
`emit_on_change`，把重复指令压缩掉。下面直接对比两者的差别。

In [ ]:
rows = []
for label, daily in (('信号驱动（默认）', False), ('每日再平衡', True)):
    strat = create_strategy('ma_cross', fast=5, slow=20, rebalance_daily=daily)
    res = engine.run(strat.generate_weights(panel), panel, label=label)
    m = res.metrics()
    rows.append(
        {
            '模式': label,
            '累计收益': f"{m['累计收益']:.2%}",
            '年化换手率': f"{m['年化双边换手率']:.1%}",
            '累计成本(元)': f"{m['累计交易成本(元)']:,.0f}",
            '成交笔数': len(res.trades),
            '年化Sharpe': f"{m['年化Sharpe']:.2f}",
        }
    )

print('同样的信号，只改「是否每日再平衡」:')
pd.DataFrame(rows)

## 4. 成交明细：亲自验证时序没有被搞错

永远不要盲信引擎。抽查成交明细，确认：

1. 成交日 = 决策日的**下一个交易日**
2. 成交价 = 该日**开盘价**
3. 成交股数是 **100 的整数倍**（A 股整手约束）

In [ ]:
print(f'共 {len(result.trades)} 笔成交，前 8 笔:')
display(result.trades.head(8))

if not result.trades.empty:
    first = result.trades.iloc[0]
    actual_open = panel.open.loc[first['date'], first['symbol']]
    print()
    print(f"第一笔成交日 : {first['date']:%Y-%m-%d}（信号日的下一个交易日）")
    print(f"成交价       : {first['price']:.4f}")
    print(f"当日开盘价   : {actual_open:.4f}  -> 一致: {np.isclose(first['price'], actual_open)}")
    print(f"成交股数     : {first['shares']:.0f}  -> 整手: {first['shares'] % 100 == 0}")

## 5. 隐形杀手二：停牌

停牌日价格与成交量都是 NaN，**当日无法成交**。若引擎忽略这一点，
就会在停牌期间「买入」一只根本买不到的股票，回测结果自然虚高。

本引擎用 `panel.tradable` 掩码屏蔽停牌交易，停牌期间的持仓用**前收盘价估值**。

In [ ]:
susp_panel = make_synthetic_panel(SYMBOLS, start=START, end=END, seed=7, suspension_prob=0.02)
susp_result = engine.run(
    create_strategy('ma_cross', fast=5, slow=20).generate_weights(susp_panel),
    susp_panel,
    label='含停牌标的',
)

tradable = susp_panel.tradable
violations = [
    row for row in susp_result.trades.itertuples() if not tradable.loc[row.date, row.symbol]
]

print(f'停牌日成交笔数（必须为 0）: {len(violations)}')
print(f'停牌样本占比: {(~tradable).to_numpy().mean():.2%}')
print(f'总成交笔数  : {len(susp_result.trades)}')

## 6. 引擎尚未建模的部分（实盘会因此变差）

诚实地说清楚边界，比假装完备更重要：

| 未建模 | 影响 |
|---|---|
| 涨跌停无法成交 | 一字板买不进也卖不出，动量策略影响最大 |
| 盘中滑点分布 | 只用固定基点近似，大单实际冲击更差 |
| 分红送股的现金流与税费 | 用前复权价格已大致抵消，但不完全精确 |
| 融资融券成本与强平 | 若策略隐含杠杆则完全未考虑 |
| 冲击成本随规模上升 | 大资金的实际容量远小于回测假设 |

**结论**：把回测结果打个折再看。行业经验是样本外实盘能拿到回测 Sharpe 的 50%~70% 已属不错。

## 7. 小结与练习

**记住四句话**

1. 收盘决策、次日开盘成交——顺序错了，后面全白算
2. 永远对比毛收益与净收益，差值就是你的成本账单
3. 每天重复的同一个指令 = 每日再平衡 = 白送手续费
4. 停牌股不能交易，只能持有并估值

**动手练习**

1. 把 `--slippage` 从 2 bps 提到 20 bps（在 `CostModel(slippage_bps=20)` 里改），
   看策略是否由盈转亏。多少 bps 是这个策略的盈亏平衡点？
2. 把初始资金从 100 万降到 5 万，观察「单笔最低佣金 5 元」的影响如何放大
3. 用 `max_participation=0.01` 打开流动性约束，看大资金下策略容量有多小
4. 打开 `plot_equity(result)` 观察回撤区间，问自己：
   如果这是真金白银，我能在回撤 -35% 时继续执行吗？